# Generating Training Data

In this notebook, the bubble dynamics model is used to generate and store training data.

In [ ]:
import sys
sys.path.append("..")
import matplotlib.pyplot as plt
import numpy as np

import wavelet_transform.utils.training_data as train
import wavelet_transform.utils.statistical_features as stats
import wavelet_transform.wpt.transform as wpt
import wavelet_transform.wpt.inverse_transform as inv

In [ ]:
sampling_rate = 50e6

In [ ]:
data = train.get_data(R_init=1e-06, frequency=11e5, pac=-60e4, sampling_rate=50e6)

data = (data - np.mean(data)) / np.std(data)
data *= np.hanning(len(data))
# data += np.random.normal(0, 0.05, len(data))

plt.plot(data)
plt.show()

tr_obj = wpt.dwpt(data, max_level=6, wavelet="db20", 
                  mode="symmetric", min_entropy=True, min_level=1
)
tr_obj.calc_frequencies(sampling_rate)
display(tr_obj.info(leaf_nodes=True))
display(tr_obj.info(0))
# print(sum(tr_obj.info(level=2)["level entropy"]))

for node in tr_obj.get_leaf_nodes():
    inverse = inv.inv_dwpt(tr_obj, node.level, nodes=node)

    # Plot inverse and node data
    fig, ax = plt.subplots(1,2)
    fig.set_size_inches(10, 3)
    ax[0].plot(inverse)
    ax[0].set_title(f"inverse transform of node{node.level, node.node_number}")
    ax[1].plot(node.data)
    ax[1].set_title("node data")
    plt.show()

### Model testing

Generating signals using the bubble dynamics model

In [ ]:
R_init = 4e-6
frequency = 3.9e6
pac = -1e5

# Read generated data or generate data and store it
data = train.get_data(R_init, frequency, pac, sampling_rate)  # uses default values for other parameters
if data is None:
    raise ValueError("data could not be generated")

plt.plot(data)
plt.title(f"R_init: {R_init}, frequency: {frequency}, pac: {pac}")
plt.show()

print(stats.kurtosis(data))

### Generating Training Data

Generating training data under a range of intensity (pac), frequency, initial bubble radius (R_init)

In [ ]:
for R_init in (i / 1E6 for i in range(1, 11)):
    for frequency in (i * 1E5 for i in range(7, 10)):
        for pac in (i * -1E4 for i in range(1, 20)):
            print(R_init, frequency, pac)     

            # Read generated data or generate it if it doesn't exist     
            data = train.get_data(R_init, frequency, pac, sampling_rate)
            # data = (data - np.mean(data)) / np.std(data)
            # data *= np.blackman(len(data))
            if data is None:
                raise ValueError("data could not be generated")

            c = max(data)
            if c > 3:
                plt.plot(data)
                plt.title(f"R_init: {R_init}, frequency: {frequency}, pac: {pac}")
                plt.show()